# Handwritten Formula to LaTeX OCR - Training and evaluating

This notebook is designed to run locally in Jupyter on Debian/Linux with NVIDIA GPU support.

**Requirements:**
- Local Jupyter kernel with project dependencies installed
- NVIDIA GPU with CUDA available (RTX 5060 Ti 16 GB is expected)
- HuggingFace account for dataset/model access

**Note:** Run Jupyter from the repository root or let the setup cell below switch into it automatically.

## 1. Setup

In [ ]:
# Check GPU
import torch
if torch.cuda.is_available():
    print(f"CUDA is available. GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("CUDA is not available. Using CPU.")

In [ ]:
# Local environment note
print("Use a prepared local virtualenv before opening this notebook.")
print("Required once outside Jupyter: pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128")
print("Then: pip install -r requirements.txt && pip install jupyter ipykernel")

# Download NLTK data
import nltk
nltk.download('punkt', quiet=True)

In [ ]:
# Clone repository
!git clone https://github.com/dmitryz1024/Multimodal_Reasoning_for_STEM.git
%cd Multimodal_Reasoning_for_STEM

In [ ]:
# Ensure we are running from the repository root and can import project modules
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

In [ ]:
# Load environment variables from .env file
from dotenv import load_dotenv
import os

# Check if .env file exists
if not os.path.exists('.env'):
    print("Warning: .env file not found. Please create a .env file with your WANDB_API_KEY and HF_TOKEN")
    print("Example .env content:")
    print("WANDB_API_KEY=your_wandb_api_key_here")
    print("HF_TOKEN=your_huggingface_token_here")
else:
    print(".env file found")

# Load environment variables
load_dotenv()

# Login to Weights & Biases
import wandb
wandb_api_key = os.getenv("WANDB_API_KEY")
if wandb_api_key:
    wandb.login(key=wandb_api_key)
    print("Successfully logged in to Weights & Biases")
else:
    print("Warning: WANDB_API_KEY not found in .env file. Wandb logging may not work.")

In [ ]:
# Verify setup
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: CUDA/GPU not available. Training will be slow on CPU.")

## 2. HuggingFace Login

In [ ]:
from huggingface_hub import login
import os

# Try to login with token from environment
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(hf_token)
    print("Logged in using HF_TOKEN environment variable")
else:
    print("Warning: HF_TOKEN not found in .env file.")
    print("Please set HF_TOKEN in your .env file or login manually using huggingface-cli")
    print("You can get a token from: https://huggingface.co/settings/tokens")

## 3. Load Datasets

In [ ]:
from datasets import load_dataset

# Load primary dataset
print("Loading LaTeX_OCR dataset...")
ds_latex_ocr = load_dataset("linxy/LaTeX_OCR", "human_handwrite")
print(ds_latex_ocr)

# Load secondary dataset (sample)
print("\nLoading MathWriting dataset...")
ds_mathwriting = load_dataset("deepcopy/MathWriting-human", split="train")
ds_mathwriting = ds_mathwriting.shuffle(seed=42).select(range(10000))  # Sample 10k
print(f"MathWriting samples: {len(ds_mathwriting)}")

## 4. Training Setup 1: LaTeX_OCR Only

In [ ]:
from src.train import TrainConfig, train

# Configuration
# Safe local configuration for RTX 5060 Ti 16 GB
config = TrainConfig(
    model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    primary_subset="human_handwrite",
    use_secondary=False,
    num_epochs=3,
    batch_size=2,
    eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    use_lora=True,
    lora_r=16,
    lora_alpha=32,
    load_in_4bit=True,
    bf16=True,
    fp16=False,
    max_length=512,
    logging_steps=1,
)

print("Configuration:")
for k, v in config.__dict__.items():
    print(f"  {k}: {v}")

# Verify device
import torch
print(f"\nDevice: GPU" if torch.cuda.is_available() else "Device: CPU")
print("Training logs will appear below: dataset loading, model loading, step progress, epoch end, and final checkpoint save.")

In [ ]:
# Monitoring info before training setup 1
from datetime import datetime
from pathlib import Path

run_name_setup1 = "sft_latex_ocr_only"
checkpoint_setup1 = Path("checkpoints") / run_name_setup1 / "final"
print("Monitoring")
print("- Current time:", datetime.now().isoformat(timespec="seconds"))
print("- Run name:", run_name_setup1)
print("- Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("- Expected checkpoint:", checkpoint_setup1.resolve())

In [ ]:
# Train with LaTeX_OCR only
print("Starting setup 1/2: SFT on LaTeX_OCR only...")
model_latex_only, processor = train(
    config=config,
    run_name="sft_latex_ocr_only",
    wandb_project="handwritten-latex-ocr"
)
print("Finished setup 1/2. Checkpoint saved to ./checkpoints/sft_latex_ocr_only/final")

## 5. Training Setup 2: Combined Dataset

In [ ]:
# Update config for combined training
print("Preparing setup 2/2: SFT on LaTeX_OCR + MathWriting...")
config.use_secondary = True
config.secondary_sample_size = 10000

run_name_setup2 = "sft_latex_ocr_mathwriting"
checkpoint_setup2 = Path("checkpoints") / run_name_setup2 / "final"
print("Monitoring")
print("- Current time:", datetime.now().isoformat(timespec="seconds"))
print("- Run name:", run_name_setup2)
print("- Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("- Expected checkpoint:", checkpoint_setup2.resolve())

# Train with combined dataset
model_combined, processor = train(
    config=config,
    run_name="sft_latex_ocr_mathwriting",
    wandb_project="handwritten-latex-ocr"
)
print("Finished setup 2/2. Checkpoint saved to ./checkpoints/sft_latex_ocr_mathwriting/final")

## 6. Evaluation

In [ ]:
from src.evaluate import run_full_evaluation, EvalConfig

eval_config = EvalConfig(
    model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    subset="human_handwrite",
    num_samples=70
)

results = run_full_evaluation(
    base_model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    checkpoint_latex_ocr="./checkpoints/sft_latex_ocr_only/final",
    checkpoint_combined="./checkpoints/sft_latex_ocr_mathwriting/final",
    config=eval_config
)

# Save results
import json
with open('evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\nResults saved to evaluation_results.json")

## 7. Upload to HuggingFace Hub

In [ ]:
# Upload LaTeX_OCR-only checkpoint to the Hugging Face Hub
!python scripts/upload_to_hub.py \
    --local_path ./checkpoints/sft_latex_ocr_only/final \
    --repo_id dmitryz1024/latex-ocr-smolvlm-latex-only

In [ ]:
# Upload combined checkpoint to the Hugging Face Hub
!python scripts/upload_to_hub.py \
    --local_path ./checkpoints/sft_latex_ocr_mathwriting/final \
    --repo_id dmitryz1024/latex-ocr-smolvlm-combined

## 8. Test Inference

In [ ]:
from src.inference import LatexOCRInference
from PIL import Image
import matplotlib.pyplot as plt

# Load fine-tuned model
engine = LatexOCRInference(
    model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    adapter_path="./checkpoints/sft_latex_ocr_only/final"
)

# Test on a few examples
test_ds = ds_latex_ocr['test']

for i in range(5):
    example = test_ds[i]
    image = example['image']
    reference = example.get('text') or example.get('latex', '')
    
    prediction = engine.predict(image)
    
    print(f"\nExample {i+1}:")
    print(f"Reference:  {reference}")
    print(f"Prediction: {prediction}")
    
    plt.figure(figsize=(6, 2))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

## 9. Download Results

In [ ]:
# Local output paths
from pathlib import Path

print("Evaluation results:", Path('evaluation_results.json').resolve())
print("Checkpoints directory:", Path('checkpoints').resolve())